In [ ]:
# Import required libraries for deep learning, image processing and visualization
import torch
import os
import matplotlib
import matplotlib.pyplot as plt
import torch.nn as nn
import PIL
import torch.optim as optim
import torchinfo
import torchvision
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from torch.utils.data import DataLoader, random_split
from torchinfo import summary
from torchvision import datasets, transforms
from tqdm import tqdm
import random
import numpy as np
import pandas as pd

In [ ]:
# Display versions of key libraries and Python
print("torch version : ", torch.__version__)
print("torchvision version : ", torchvision.__version__)
print("torchinfo version : ", torchinfo.__version__)
print("numpy version : ", np.__version__)
print("matplotlib version : ", matplotlib.__version__)
print("PIL version : ", PIL.__version__)

!python --version

In [ ]:
# Set device to GPU if available, otherwise use CPU
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
print(f"device is: {device}")

In [ ]:
# Define path to the clothing dataset
data = '/kaggle/input/e-commerce-mens-clothing-dataset'
data_dir = os.path.join(data,"dataset_clean")

In [ ]:
# List all clothing categories (classes) in the dataset
classes = os.listdir(data_dir)
print(f"List of classes are:", classes)

In [ ]:
# Custom transformation to convert grayscale images to RGB format
class ConvertToRGB(object):
    def __call__(self,img):
        if img.mode != "RGB":
            img = img.convert("RGB")
        return img

In [ ]:
# Function to compute mean and standard deviation for dataset normalization
def get_mean_std(loader):
    """Computes the mean and standard deviation of image data.

    Input: a `DataLoader` producing tensors of shape [batch_size, channels, pixels_x, pixels_y]
    Output: the mean of each channel as a tensor, the standard deviation of each channel as a tensor
            formatted as a tuple (means[channels], std[channels])"""

    channels_sum, channels_squared_sum, num_batches = 0, 0, 0
    for data, _ in tqdm(loader, desc="Computing mean and std", leave=False):
        # Sum means and squared values for each channel
        channels_sum += torch.mean(data, dim=[0, 2, 3])
        channels_squared_sum += torch.mean(data**2, dim=[0, 2, 3])
        num_batches += 1
    
    # Calculate mean and std using the formula: std = sqrt(E[X^2] - (E[X])^2)
    mean = channels_sum / num_batches
    std = (channels_squared_sum / num_batches - mean**2) ** 0.5

    return mean, std

In [ ]:
# Define basic image transformations: convert to RGB, resize, and convert to tensor
transform = transforms.Compose([
    ConvertToRGB(),
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [ ]:
# Create dataset using ImageFolder structure (automatically loads images and assigns class labels)
dataset = datasets.ImageFolder(root=data_dir,transform=transform)

In [ ]:
# Create dataloader for batching and shuffling images
dataset_loader = DataLoader(dataset,batch_size=32,shuffle=True)

In [ ]:
# Verify batch shape to ensure data loads correctly
batch_shape = next(iter(dataset_loader))[0].shape
print("Getting batches of shape:", batch_shape)

In [ ]:
# Function to visualize sample images from a specific clothing category
def sample_images(data_path, classname):
    # Gets the files in the directory
    class_dir = os.path.join(data_path, classname)
    if not os.path.exists(class_dir):
        return "Invalid directory"
    image_list = os.listdir(class_dir)
    if len(image_list) < 4:
        return "Not enough images in folder"

    # Pick four random images
    images_sample = random.sample(image_list, 4)

    # Plot them in a grid
    plt.figure(figsize=(20, 20))
    for i in range(4):
        img_loc = os.path.join(class_dir, images_sample[i])
        img = PIL.Image.open(img_loc)
        plt.subplot(1, 4, i + 1)
        plt.imshow(img)
        plt.axis("off")

In [ ]:
# Calculate mean and standard deviation for normalization
mean, std = get_mean_std(dataset_loader)

print(f"Mean: {mean}")
print(f"Standard deviation: {std}")

In [ ]:
# Visualize sample images from the men_cargos class
sample_images(data_dir,"men_cargos")

In [ ]:
# Define normalized transforms with mean and std for proper image preprocessing
transform_norm = transforms.Compose([
    ConvertToRGB(),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean,std=std)
])

In [ ]:
# Create normalized dataset and dataloader
dataset_norm = datasets.ImageFolder(root=data_dir,transform=transform_norm)
dataloader_norm = DataLoader(dataset_norm,batch_size=32,shuffle=True)

In [ ]:
# Verify normalized batch shape
batch_shape = next(iter(dataloader_norm))[0].shape
print("Getting batches of shape:", batch_shape)

In [ ]:
# Split dataset into training (80%), validation (10%), and test (10%) sets
train_len = int(0.8 * len(dataset_norm))
val_len = int(0.1 * len(dataset_norm))
test_len = len(dataset_norm) - train_len - val_len

print(f"Length of training set will be: {train_len}, "
      f"validation set will be: {val_len} "
      f"and testing set will be {test_len}.")

train_dataset, val_dataset, test_dataset = random_split(
    dataset_norm,
    [train_len, val_len, test_len]
)

In [ ]:
# Function to count class distribution in a dataset
from collections import Counter 
def class_counts(dataset):
    # Count samples for each class
    c = Counter(x[1] for x in tqdm(dataset))
    try:
        class_to_index = dataset.class_to_idx
    except AttributeError:
        class_to_index = dataset.dataset.class_to_idx
    return pd.Series({cat: c[idx] for cat, idx in class_to_index.items()})

In [ ]:
# Get class distribution for training set
train_classes = class_counts(train_dataset)

In [ ]:
# Visualize training set class distribution
train_classes.plot(kind="bar")

In [ ]:
# Get and visualize validation set class distribution
val_classes = class_counts(val_dataset)
val_classes.plot(kind="bar")

In [ ]:
# Create data loaders for training and validation
train_loader = DataLoader(train_dataset, batch_size=32,shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32,shuffle=False)

In [ ]:
# Define custom CNN model with 3 convolutional blocks and 2 fully connected layers
model = nn.Sequential(
    # First convolutional block: 3 input channels -> 16 filters
    nn.Conv2d(3, 16, 3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    
    # Second convolutional block: 16 filters -> 32 filters
    nn.Conv2d(16, 32, 3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    
    # Third convolutional block: 32 filters -> 64 filters
    nn.Conv2d(32, 64, 3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    
    # Flatten for fully connected layers
    nn.Flatten(),
    
    # First fully connected layer with dropout for regularization
    nn.Dropout(0.5),
    nn.Linear(64 * 28 * 28, 128),  
    nn.ReLU(),
    nn.Dropout(0.5),
    
    # Output layer: 8 classes for clothing categories
    nn.Linear(128, 8)
)

In [ ]:
# Move model to GPU/CPU device
model.to(device)

In [ ]:
# Define loss function and optimizer for custom CNN training
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Function to calculate accuracy: percentage of correct predictions
def accuracy_fn(outputs, labels):
    _, preds = torch.max(outputs, 1)
    correct = (preds == labels).sum().item()
    return correct / len(labels)

# Early stopping class to prevent overfitting
class EarlyStopping:
    def __init__(self, patience=3, delta=0.0):
        """
        Args:
            patience: Number of epochs to wait for improvement before stopping
            delta: Minimum change in validation loss to qualify as an improvement
        """
        self.patience = patience
        self.delta = delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
    
    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss < self.best_loss - self.delta:
            # Improvement found, reset counter
            self.best_loss = val_loss
            self.counter = 0
        else:
            # No improvement, increment counter
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        
        return self.early_stop

In [ ]:
# Train custom CNN model with early stopping to prevent overfitting
epochs = 10
early_stopping = EarlyStopping(patience=3, delta=0.001)

for epoch in range(epochs):
    
    # ===== TRAINING PHASE =====
    model.train()  # Set model to training mode
    train_loss = 0
    train_acc = 0
    
    for images, labels in train_loader:
        # Move data to device
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimization
        loss.backward()
        optimizer.step()
        
        # Accumulate loss and accuracy
        train_loss += loss.item()
        train_acc += accuracy_fn(outputs, labels)
    
    # Calculate average training metrics
    train_loss /= len(train_loader)
    train_acc /= len(train_loader)
    
    
    # ===== VALIDATION PHASE =====
    model.eval()  # Set model to evaluation mode
    val_loss = 0
    val_acc = 0
    
    # No gradient computation during validation
    with torch.no_grad():
        for images, labels in val_loader:
            # Move data to device
            images = images.to(device)
            labels = labels.to(device)
            
            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            # Accumulate loss and accuracy
            val_loss += loss.item()
            val_acc += accuracy_fn(outputs, labels)
    
    # Calculate average validation metrics
    val_loss /= len(val_loader)
    val_acc /= len(val_loader)
    
    # Print epoch statistics
    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    
    # Check for early stopping
    if early_stopping(val_loss):
        print(f"Early stopping triggered at epoch {epoch+1}")
        break

In [ ]:
# Create dataloader for test set (no shuffling for consistent evaluation)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle = False)

In [ ]:
# Evaluate custom CNN model on test set
model.eval()  # Set model to evaluation mode
test_loss = 0
test_acc = 0

# No gradient computation during testing
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Accumulate loss and accuracy
        test_loss += loss.item()
        test_acc += accuracy_fn(outputs, labels)

# Calculate average test metrics
test_loss /= len(test_loader)
test_acc /= len(test_loader)

print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

# Transfer Learning with ResNet50

Pre-trained ResNet50 model fine-tuned on our clothing classification dataset

In [ ]:
# Load pre-trained ResNet50 model and configure for transfer learning
from torchvision import models

# Load pre-trained ResNet50 (trained on ImageNet)
resnet50 = models.resnet50(pretrained=True)

# Freeze all convolutional layers to preserve learned features
for param in resnet50.parameters():
    param.requires_grad = False

# Unfreeze and replace final classification layer for our 8 clothing classes
num_classes = len(classes)
resnet50.fc = nn.Linear(resnet50.fc.in_features, num_classes)

# Move model to GPU/CPU device
resnet50 = resnet50.to(device)

print("ResNet50 Model Loaded")
print(f"Number of trainable parameters: {sum(p.numel() for p in resnet50.parameters() if p.requires_grad)}")

In [ ]:
# Define loss function and optimizer for ResNet50 transfer learning
# Only optimize the final layer parameters (frozen feature extraction)
criterion_resnet = nn.CrossEntropyLoss()
optimizer_resnet = torch.optim.Adam(resnet50.fc.parameters(), lr=0.001)

In [ ]:
# Train ResNet50 with transfer learning and early stopping
epochs_resnet = 10
# Lists to track metrics for plotting
train_losses_resnet = []
val_losses_resnet = []
train_accs_resnet = []
val_accs_resnet = []
# Initialize early stopping for ResNet50
early_stopping_resnet = EarlyStopping(patience=3, delta=0.001)

for epoch in range(epochs_resnet):
    
    # ===== TRAINING PHASE =====
    resnet50.train()  # Set model to training mode
    train_loss = 0
    train_acc = 0
    
    # Train with progress bar
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs_resnet} - Train"):
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass
        optimizer_resnet.zero_grad()
        outputs = resnet50(images)
        loss = criterion_resnet(outputs, labels)
        
        # Backward pass and optimization
        loss.backward()
        optimizer_resnet.step()
        
        # Accumulate loss and accuracy
        train_loss += loss.item()
        train_acc += accuracy_fn(outputs, labels)
    
    # Calculate average training metrics
    train_loss /= len(train_loader)
    train_acc /= len(train_loader)
    train_losses_resnet.append(train_loss)
    train_accs_resnet.append(train_acc)
    
    # ===== VALIDATION PHASE =====
    resnet50.eval()  # Set model to evaluation mode
    val_loss = 0
    val_acc = 0
    
    # No gradient computation during validation
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            # Forward pass
            outputs = resnet50(images)
            loss = criterion_resnet(outputs, labels)
            
            # Accumulate loss and accuracy
            val_loss += loss.item()
            val_acc += accuracy_fn(outputs, labels)
    
    # Calculate average validation metrics
    val_loss /= len(val_loader)
    val_acc /= len(val_loader)
    val_losses_resnet.append(val_loss)
    val_accs_resnet.append(val_acc)
    
    # Print epoch statistics
    print(f"Epoch [{epoch+1}/{epochs_resnet}] "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    
    # Check for early stopping
    if early_stopping_resnet(val_loss):
        print(f"Early stopping triggered at epoch {epoch+1}")
        break

In [ ]:
# Evaluate ResNet50 on test set and collect predictions for confusion matrix
resnet50.eval()  # Set model to evaluation mode
test_loss_resnet = 0
test_acc_resnet = 0
# Store all predictions and labels for confusion matrix
all_preds_resnet = []
all_labels_resnet = []

# No gradient computation during testing
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass
        outputs = resnet50(images)
        loss = criterion_resnet(outputs, labels)
        
        # Accumulate loss and accuracy
        test_loss_resnet += loss.item()
        test_acc_resnet += accuracy_fn(outputs, labels)
        
        # Collect predictions for confusion matrix
        _, preds = torch.max(outputs, 1)
        all_preds_resnet.extend(preds.cpu().numpy())
        all_labels_resnet.extend(labels.cpu().numpy())

# Calculate average test metrics
test_loss_resnet /= len(test_loader)
test_acc_resnet /= len(test_loader)

# Print test results
print(f"\n{'='*60}")
print(f"ResNet50 Transfer Learning - Test Results")
print(f"{'='*60}")
print(f"Test Loss: {test_loss_resnet:.4f}")
print(f"Test Accuracy: {test_acc_resnet:.4f}")
print(f"{'='*60}")

In [ ]:
# Plot ResNet50 training curves for loss and accuracy
plt.figure(figsize=(14, 5))

# Plot loss curves
plt.subplot(1, 2, 1)
plt.plot(train_losses_resnet, label='Train Loss', linewidth=2)
plt.plot(val_losses_resnet, label='Val Loss', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('ResNet50 - Loss curves')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot accuracy curves
plt.subplot(1, 2, 2)
plt.plot(train_accs_resnet, label='Train Accuracy', linewidth=2)
plt.plot(val_accs_resnet, label='Val Accuracy', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('ResNet50 - Accuracy curves')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Generate and visualize confusion matrix for ResNet50 predictions
cm_resnet = confusion_matrix(all_labels_resnet, all_preds_resnet)
disp_resnet = ConfusionMatrixDisplay(confusion_matrix=cm_resnet, display_labels=classes)

plt.figure(figsize=(10, 8))
disp_resnet.plot(cmap='Blues')
plt.title('ResNet50 - Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Compare performance between custom CNN and ResNet50 transfer learning
print(f"\n{'='*60}")
print(f"MODEL PERFORMANCE COMPARISON")
print(f"{'='*60}")
print(f"Custom CNN Test Accuracy:  {test_acc:.4f}")
print(f"ResNet50 Test Accuracy:    {test_acc_resnet:.4f}")
print(f"{'='*60}")
print(f"Improvement: {(test_acc_resnet - test_acc)*100:.2f}%")
print(f"{'='*60}")